In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt

In [3]:
res = pd.read_csv("data/1976-2024-house.tab")
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,candidate,party,writein,mode,candidatevotes,totalvotes,unofficial,version,fusion_ticket
0,1976,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,False,False,BILL DAVENPORT,DEMOCRAT,False,TOTAL,58906,157170,False,20250910,False
1,1976,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,False,False,JACK EDWARDS,REPUBLICAN,False,TOTAL,98257,157170,False,20250910,False
2,1976,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,False,False,WRITEIN,NaN,True,TOTAL,7,157170,False,20250910,False
3,1976,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,False,False,J CAROLE KEAHEY,DEMOCRAT,False,TOTAL,66288,156362,False,20250910,False
4,1976,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,False,False,"WILLIAM L \""BILL\"" DICKINSON",REPUBLICAN,False,TOTAL,90069,156362,False,20250910,False


In [4]:
res.columns.values

array(['year', 'state', 'state_po', 'state_fips', 'state_cen', 'state_ic',
       'office', 'district', 'stage', 'runoff', 'special', 'candidate',
       'party', 'writein', 'mode', 'candidatevotes', 'totalvotes',
       'unofficial', 'version', 'fusion_ticket'], dtype=object)

In [5]:
res = res[(res['year'] >= 2018) & (res['stage'] == 'GEN') & (res['mode'] == 'TOTAL')]
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,candidate,party,writein,mode,candidatevotes,totalvotes,unofficial,version,fusion_ticket
28277,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,BRADLEY BYRNE,REPUBLICAN,False,TOTAL,153228,242617,False,20250910,False
28278,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,ROBERT KENNEDY JR,DEMOCRAT,False,TOTAL,89226,242617,False,20250910,False
28279,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,WRITEIN,NaN,True,TOTAL,163,242617,False,20250910,False
28280,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,False,MARTHA ROBY,REPUBLICAN,False,TOTAL,138879,226230,False,20250910,False
28281,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,False,TABITHA ISNER,DEMOCRAT,False,TOTAL,86931,226230,False,20250910,False


In [6]:
res['party'].value_counts()

party
DEMOCRAT                   1711
REPUBLICAN                 1678
LIBERTARIAN                 389
INDEPENDENT                 184
CONSERVATIVE                 88
                           ... 
WRITE-IN (UNAFFILIATED)       1
POPULIST PARTY                1
ALOHA AINA PARTY              1
AMERICAN SHOPPING PARTY       1
REPUBLICAN, LIBERTARIAN       1
Name: count, Length: 131, dtype: int64

In [7]:
# Focus on two party vote share in the model - no need to wrangle with third parties for now
res = res[res['party'].isin(['DEMOCRAT', 'REPUBLICAN'])]
res.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,candidate,party,writein,mode,candidatevotes,totalvotes,unofficial,version,fusion_ticket
28277,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,BRADLEY BYRNE,REPUBLICAN,False,TOTAL,153228,242617,False,20250910,False
28278,2018,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,NaN,False,ROBERT KENNEDY JR,DEMOCRAT,False,TOTAL,89226,242617,False,20250910,False
28280,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,False,MARTHA ROBY,REPUBLICAN,False,TOTAL,138879,226230,False,20250910,False
28281,2018,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,NaN,False,TABITHA ISNER,DEMOCRAT,False,TOTAL,86931,226230,False,20250910,False
28283,2018,ALABAMA,AL,1,63,41,US HOUSE,3,GEN,NaN,False,MALLORY HAGAN,DEMOCRAT,False,TOTAL,83996,231915,False,20250910,False


In [8]:
res.shape

(3389, 20)

In [9]:
# aggfunc is sum to account for races where two or more candidates from the same party are running in the general,
# we sum the votes of these candidates together as part of calculating two-party vote share
data = pd.pivot_table(data=res, values='candidatevotes', columns=['party'], index=['year', 'state', 'state_po', 'special', 'district'], aggfunc='sum').reset_index()
data.head()

party,year,state,state_po,special,district,DEMOCRAT,REPUBLICAN
0,2018,ALABAMA,AL,False,1,89226.0,153228.0
1,2018,ALABAMA,AL,False,2,86931.0,138879.0
2,2018,ALABAMA,AL,False,3,83996.0,147770.0
3,2018,ALABAMA,AL,False,4,46492.0,184255.0
4,2018,ALABAMA,AL,False,5,101388.0,159063.0


In [10]:
data.duplicated(subset=['year', 'state', 'special', 'district']).any() # np.False_ --> no duplicates

np.False_

In [11]:
# Get candidates

def get_cands(year, state, special, district, party):
    df = res[
        (res['year'] == year) &
        (res['state'] == state) &
        (res['special'] == special) &
        (res['district'] == district) &
        (res['party'] == party)
    ]
    if df.shape[0] == 1:
        return df['candidate'].values[0]
    else:
        return repr(list(df['candidate'].values))

def get_totvotes(year, state, special, district):
    df = res[
        (res['year'] == year) &
        (res['state'] == state) &
        (res['special'] == special) &
        (res['district'] == district)
    ]
    return df['totalvotes'].values[0]

In [12]:
get_cands(2018, 'ALABAMA', False, 1, 'DEMOCRAT') # conventional

'ROBERT KENNEDY JR'

In [13]:
get_cands(2024, 'CALIFORNIA', False, 12, 'DEMOCRAT') # two people same party

"['JENNIFER TRAN', 'LATEEFAH SIMON']"

In [14]:
get_cands(2024, 'Massachusetts'.upper(), False, 4, 'REPUBLICAN') # running unopposed

'[]'

In [15]:
def get_dem_cand(year, state, special, district):
    return get_cands(year, state, special, district, 'DEMOCRAT')

def get_rep_cand(year, state, special, district):
    return get_cands(year, state, special, district, 'REPUBLICAN')

In [16]:
data['dem_cand'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_dem_cand(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data['rep_cand'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_rep_cand(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data.head()

party,year,state,state_po,special,district,DEMOCRAT,REPUBLICAN,dem_cand,rep_cand
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,ROBERT KENNEDY JR,BRADLEY BYRNE
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,TABITHA ISNER,MARTHA ROBY
2,2018,ALABAMA,AL,False,3,83996.0,147770.0,MALLORY HAGAN,MIKE ROGERS
3,2018,ALABAMA,AL,False,4,46492.0,184255.0,LEE AUMAN,ROBERT ADERHOLT
4,2018,ALABAMA,AL,False,5,101388.0,159063.0,PETER JOFFRION,MO BROOKS


In [17]:
data = data.rename({'DEMOCRAT': 'dem', 'REPUBLICAN': 'rep'}, axis=1)
data.head()

party,year,state,state_po,special,district,dem,rep,dem_cand,rep_cand
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,ROBERT KENNEDY JR,BRADLEY BYRNE
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,TABITHA ISNER,MARTHA ROBY
2,2018,ALABAMA,AL,False,3,83996.0,147770.0,MALLORY HAGAN,MIKE ROGERS
3,2018,ALABAMA,AL,False,4,46492.0,184255.0,LEE AUMAN,ROBERT ADERHOLT
4,2018,ALABAMA,AL,False,5,101388.0,159063.0,PETER JOFFRION,MO BROOKS


In [18]:
data.shape

(1742, 9)

In [19]:
data['totalvotes'] = data[['year', 'state', 'special', 'district']].apply(lambda x: get_totvotes(x['year'], x['state'], x['special'],
                                                                                              x['district']), axis=1)
data.head()

party,year,state,state_po,special,district,dem,rep,dem_cand,rep_cand,totalvotes
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,ROBERT KENNEDY JR,BRADLEY BYRNE,242617
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,TABITHA ISNER,MARTHA ROBY,226230
2,2018,ALABAMA,AL,False,3,83996.0,147770.0,MALLORY HAGAN,MIKE ROGERS,231915
3,2018,ALABAMA,AL,False,4,46492.0,184255.0,LEE AUMAN,ROBERT ADERHOLT,230969
4,2018,ALABAMA,AL,False,5,101388.0,159063.0,PETER JOFFRION,MO BROOKS,260673


In [20]:
data.shape

(1742, 10)

In [21]:
data['2party_votes'] = data['dem'] + data['rep']
data.head(2)

party,year,state,state_po,special,district,dem,rep,dem_cand,rep_cand,totalvotes,2party_votes
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,ROBERT KENNEDY JR,BRADLEY BYRNE,242617,242454.0
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,TABITHA ISNER,MARTHA ROBY,226230,225810.0


In [22]:
data['dem_cand'] = data['dem_cand'].str.title()
data['rep_cand'] = data['rep_cand'].str.title()
data.head(2)

party,year,state,state_po,special,district,dem,rep,dem_cand,rep_cand,totalvotes,2party_votes
0,2018,ALABAMA,AL,False,1,89226.0,153228.0,Robert Kennedy Jr,Bradley Byrne,242617,242454.0
1,2018,ALABAMA,AL,False,2,86931.0,138879.0,Tabitha Isner,Martha Roby,226230,225810.0


In [23]:
data['state'] = data['state'].str.title()
data.head(2)

party,year,state,state_po,special,district,dem,rep,dem_cand,rep_cand,totalvotes,2party_votes
0,2018,Alabama,AL,False,1,89226.0,153228.0,Robert Kennedy Jr,Bradley Byrne,242617,242454.0
1,2018,Alabama,AL,False,2,86931.0,138879.0,Tabitha Isner,Martha Roby,226230,225810.0


In [24]:
# two-party vote share
data['dem_pct_2p'] = data['dem'] / data['2party_votes'] * 100
data['rep_pct_2p'] = data['rep'] / data['2party_votes'] * 100
data.head(2)

party,year,state,state_po,special,district,dem,rep,dem_cand,rep_cand,totalvotes,2party_votes,dem_pct_2p,rep_pct_2p
0,2018,Alabama,AL,False,1,89226.0,153228.0,Robert Kennedy Jr,Bradley Byrne,242617,242454.0,36.801208,63.198792
1,2018,Alabama,AL,False,2,86931.0,138879.0,Tabitha Isner,Martha Roby,226230,225810.0,38.497409,61.502591


In [25]:
fec_webl_colnames = ["CAND_ID", "CAND_NAME", "CAND_ICI", "PTY_CD", "CAND_PTY_AFFILIATION", "TTL_RECEIPTS", "TRANS_FROM_AUTH", "TTL_DISB", "TRANS_TO_AUTH", "COH_BOP", "COH_COP", "CAND_CONTRIB", "CAND_LOANS", "OTHER_LOANS", "CAND_LOAN_REPAY", "OTHER_LOAN_REPAY", "DEBTS_OWED_BY", "TTL_INDIV_CONTRIB", "CAND_OFFICE_ST", "CAND_OFFICE_DISTRICT", "SPEC_ELECTION", "PRIM_ELECTION", "RUN_ELECTION", "GEN_ELECTION", "GEN_ELECTION_PRECENT", "OTHER_POL_CMTE_CONTRIB", "POL_PTY_CONTRIB", "CVG_END_DT", "INDIV_REFUNDS", "CMTE_REFUNDS"]

In [26]:
# From: https://github.com/markjrieke/2022-midterm-forecasts/tree/main
rieke = pd.read_csv('data/rieke_historical_results_2018_2020.csv')
rieke.head()

,cycle,race,state,seat,candidate_name_DEM,candidate_name_REP,dem_votes,rep_votes,dem_incumbent,rep_incumbent,prev_seat
0,2018,House,Alabama,District 1,Robert Kennedy Jr.,Bradley Byrne,89226,153228,n,y,rep
1,2018,House,Alabama,District 2,Tabitha Isner,Martha Roby,86931,138879,n,y,rep
2,2018,House,Alabama,District 3,Mallory Hagan,Mike Rogers,83996,147770,n,y,rep
3,2018,House,Alabama,District 4,Lee Auman,Robert Aderholt,46492,184255,n,y,rep
4,2018,House,Alabama,District 5,Peter Joffrion,Mo Brooks,101388,159063,n,y,rep


In [27]:
rieke_22 = pd.read_csv('data/rieke_historical_results_2022.csv', encoding='latin-1')
rieke_22.head()

,cycle,race,state,seat,candidate_name_DEM,candidate_name_REP,dem_incumbent,rep_incumbent,pending
0,2022,House,Alabama,District 1,Remrey,Carl,n,y,NaN
1,2022,House,Alabama,District 2,Harvey-Hall,Moore,n,y,NaN
2,2022,House,Alabama,District 3,Veasey,Rogers,n,y,NaN
3,2022,House,Alabama,District 4,Neighbors,Aderholt,n,y,NaN
4,2022,House,Alabama,District 5,Warner-Stanton,Strong,n,n,NaN


In [28]:
rieke = pd.concat([rieke, rieke_22], axis=0)
rieke.head()

,cycle,race,state,seat,candidate_name_DEM,candidate_name_REP,dem_votes,rep_votes,dem_incumbent,rep_incumbent,prev_seat,pending
0,2018,House,Alabama,District 1,Robert Kennedy Jr.,Bradley Byrne,89226.0,153228.0,n,y,rep,NaN
1,2018,House,Alabama,District 2,Tabitha Isner,Martha Roby,86931.0,138879.0,n,y,rep,NaN
2,2018,House,Alabama,District 3,Mallory Hagan,Mike Rogers,83996.0,147770.0,n,y,rep,NaN
3,2018,House,Alabama,District 4,Lee Auman,Robert Aderholt,46492.0,184255.0,n,y,rep,NaN
4,2018,House,Alabama,District 5,Peter Joffrion,Mo Brooks,101388.0,159063.0,n,y,rep,NaN


In [29]:
rieke.tail()

,cycle,race,state,seat,candidate_name_DEM,candidate_name_REP,dem_votes,rep_votes,dem_incumbent,rep_incumbent,prev_seat,pending
501,2022,Governor,Tennessee,Governor,Martin,Lee,NaN,NaN,n,y,NaN,NaN
502,2022,Governor,Alabama,Governor,Flowers,Ivey,NaN,NaN,n,y,NaN,NaN
503,2022,Governor,South Dakota,Governor,Smith,Noem,NaN,NaN,n,y,NaN,NaN
504,2022,Governor,Idaho,Governor,Heidt,Little,NaN,NaN,n,y,NaN,NaN
505,2022,Governor,Wyoming,Governor,Livingston,Gordon,NaN,NaN,n,y,NaN,NaN


In [30]:
rieke['dem_incumbent'] = rieke['dem_incumbent'].replace({'n': False, 'y': True})
rieke['rep_incumbent'] = rieke['rep_incumbent'].replace({'n': False, 'y': True})

In [31]:
rieke = rieke.rename({'dem_incumbent': 'dem_inc', 'rep_incumbent': 'rep_inc', 'seat': 'district',
                     'cycle': 'year'}, axis=1)

In [32]:
rieke = rieke[~rieke['district'].isin(['Class I', 'Class II', 'Class III', 'Governor'])]
rieke['district'] = rieke['district'].str.lstrip('District ').astype(int)
rieke.head(3)

,year,race,state,district,candidate_name_DEM,candidate_name_REP,dem_votes,rep_votes,dem_inc,rep_inc,prev_seat,pending
0,2018,House,Alabama,1,Robert Kennedy Jr.,Bradley Byrne,89226.0,153228.0,False,True,rep,NaN
1,2018,House,Alabama,2,Tabitha Isner,Martha Roby,86931.0,138879.0,False,True,rep,NaN
2,2018,House,Alabama,3,Mallory Hagan,Mike Rogers,83996.0,147770.0,False,True,rep,NaN


In [33]:
data.shape

(1742, 13)

In [34]:
data = pd.merge(left=data, right=rieke[['year', 'state', 'district', 'dem_inc', 'rep_inc']], on=['year', 'state', 'district'],
               how='left')
data.head(3)

,year,state,state_po,special,district,dem,rep,dem_cand,rep_cand,totalvotes,2party_votes,dem_pct_2p,rep_pct_2p,dem_inc,rep_inc
0,2018,Alabama,AL,False,1,89226.0,153228.0,Robert Kennedy Jr,Bradley Byrne,242617,242454.0,36.801208,63.198792,False,True
1,2018,Alabama,AL,False,2,86931.0,138879.0,Tabitha Isner,Martha Roby,226230,225810.0,38.497409,61.502591,False,True
2,2018,Alabama,AL,False,3,83996.0,147770.0,Mallory Hagan,Mike Rogers,231915,231766.0,36.241727,63.758273,False,True


In [35]:
data.shape

(1742, 15)

In [36]:
## For now, fill in True for NaN entries; use Wikipedia and Excel to discern and fill in entries where it's False (for dem_inc, rep_inc)
data['dem_inc'] = data['dem_inc'].fillna(True)
data['rep_inc'] = data['rep_inc'].fillna(True)
data['dem_inc'].isna().any()

np.False_

In [37]:
data['dem_pct_2p'] = 100 * data['dem'] / data['2party_votes']
data['rep_pct_2p'] = 100 * data['rep'] / data['2party_votes']
data.head(3)

,year,state,state_po,special,district,dem,rep,dem_cand,rep_cand,totalvotes,2party_votes,dem_pct_2p,rep_pct_2p,dem_inc,rep_inc
0,2018,Alabama,AL,False,1,89226.0,153228.0,Robert Kennedy Jr,Bradley Byrne,242617,242454.0,36.801208,63.198792,False,True
1,2018,Alabama,AL,False,2,86931.0,138879.0,Tabitha Isner,Martha Roby,226230,225810.0,38.497409,61.502591,False,True
2,2018,Alabama,AL,False,3,83996.0,147770.0,Mallory Hagan,Mike Rogers,231915,231766.0,36.241727,63.758273,False,True


In [38]:
webl18 = pd.read_table('data/fec/webl18.txt', sep='|', names=fec_webl_colnames)
webl20 = pd.read_table('data/fec/webl20.txt', sep='|', names=fec_webl_colnames)
webl22 = pd.read_table('data/fec/webl22.txt', sep='|', names=fec_webl_colnames)
webl24 = pd.read_table('data/fec/webl24.txt', sep='|', names=fec_webl_colnames)
webl24.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,...,SPEC_ELECTION,PRIM_ELECTION,RUN_ELECTION,GEN_ELECTION,GEN_ELECTION_PRECENT,OTHER_POL_CMTE_CONTRIB,POL_PTY_CONTRIB,CVG_END_DT,INDIV_REFUNDS,CMTE_REFUNDS
0,H2AK01158,"PELTOLA, MARY",I,1,DEM,13443537.46,951851.88,14050828.27,0.00,691260.30,...,NaN,NaN,NaN,NaN,NaN,1615986.30,9969.28,12/31/2024,161309.36,5625.0
1,H2AK01083,"BEGICH, NICHOLAS III",C,2,REP,2810467.65,176570.41,2747371.58,17659.66,41233.99,...,NaN,NaN,NaN,NaN,NaN,318750.00,5000.00,12/31/2024,23031.99,0.0
2,H4AK00156,"DAHLSTROM, NANCY",C,2,REP,996163.60,435712.04,790351.61,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,262916.02,0.00,12/31/2024,3218.30,0.0
3,H4AL01255,"HOLMES, THOMAS BETHUNE MR.",C,1,DEM,17698.86,0.00,16817.50,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,2002.63,0.00,12/31/2024,0.00,0.0
4,H0AL01055,"CARL, JERRY LEE, JR",I,2,REP,2246839.19,547807.76,2631446.59,27316.59,453897.82,...,NaN,NaN,NaN,NaN,NaN,634500.00,0.00,12/31/2024,193499.00,84500.0


In [39]:
fec_cn_colnames = ["CAND_ID", "CAND_NAME", "CAND_PTY_AFFILIATION", "CAND_ELECTION_YR", "CAND_OFFICE_ST", "CAND_OFFICE", "CAND_OFFICE_DISTRICT", "CAND_ICI", "CAND_STATUS", "CAND_PCC", "CAND_ST1", "CAND_ST2", "CAND_CITY", "CAND_ST", "CAND_ZIP"]

cn18 = pd.read_table('data/fec/cn18.txt', sep='|', names=fec_cn_colnames)
cn20 = pd.read_table('data/fec/cn20.txt', sep='|', names=fec_cn_colnames)
cn22 = pd.read_table('data/fec/cn22.txt', sep='|', names=fec_cn_colnames)
cn24 = pd.read_table('data/fec/cn24.txt', sep='|', names=fec_cn_colnames)
cn24.head()

,CAND_ID,CAND_NAME,CAND_PTY_AFFILIATION,CAND_ELECTION_YR,CAND_OFFICE_ST,CAND_OFFICE,CAND_OFFICE_DISTRICT,CAND_ICI,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
0,H0AK00105,"LAMB, THOMAS",NNE,2020,AK,H,0.0,C,N,C00607515,1861 W LAKE LUCILLE DR,NaN,WASILLA,AK,99654
1,H0AL01055,"CARL, JERRY LEE, JR",REP,2024,AL,H,1.0,I,C,C00697789,PO BOX 852138,NaN,MOBILE,AL,36685
2,H0AL01097,"AVERHART, JAMES",DEM,2024,AL,H,2.0,C,C,C00708867,811 SPRINGHILL AV,NaN,MOBILE,AL,36602
3,H0AL02087,"ROBY, MARTHA",REP,2020,AL,H,2.0,I,P,C00462143,NaN,NaN,MONTGOMERY,NaN,NaN
4,H0AL02137,"DISMUKES, WILL",REP,2020,AL,H,2.0,O,P,C00714337,PO BOX 6811188,NaN,PRATTVILLE,AL,36068


In [48]:
df_test = pd.merge(left=webl24, right=cn24, on='CAND_ID', how='inner')
df_test = df_test[[col for col in df_test.columns.values if ('_y' not in col)]]
df_test.columns = df_test.columns.str.strip('_x')
df_test.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,...,CMTE_REFUNDS,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP
0,H2AK01158,"PELTOLA, MARY",I,1,DEM,13443537.46,951851.88,14050828.27,0.00,691260.30,...,5625.0,2024,H,C,C00812388,810 N STREET,SUITE 301,ANCHORAGE,AK,99501
1,H2AK01083,"BEGICH, NICHOLAS III",C,2,REP,2810467.65,176570.41,2747371.58,17659.66,41233.99,...,0.0,2024,H,C,C00792341,PO BOX 671710,NaN,CHUGIAK,AK,99567
2,H4AK00156,"DAHLSTROM, NANCY",C,2,REP,996163.60,435712.04,790351.61,0.00,0.00,...,0.0,2024,H,C,C00856716,PO BOX 242442,NaN,ANCHORAGE,AK,99524
3,H4AL01255,"HOLMES, THOMAS BETHUNE MR.",C,1,DEM,17698.86,0.00,16817.50,0.00,0.00,...,0.0,2024,H,C,C00866939,"2117 CHARINGWOOD DRIVE WEST, MOBIL",NaN,MOBILE,AL,366952916
4,H0AL01055,"CARL, JERRY LEE, JR",I,2,REP,2246839.19,547807.76,2631446.59,27316.59,453897.82,...,84500.0,2024,H,C,C00697789,PO BOX 852138,NaN,MOBILE,AL,36685


In [58]:
def wrangle_fec(webl, cn, year):
    webl['cand_name_lst'] = webl['CAND_NAME'].str.split(',')
    cn['cand_name_lst'] = cn['CAND_NAME'].str.split(',')

    def refactor_str(cand_lst):
        if len(cand_lst) == 3:
            return cand_lst[1] + ' ' + cand_lst[0] + ' ' + cand_lst[2]
        elif len(cand_lst) == 1:
            return cand_lst[0]
        else:
            return cand_lst[1] + ' ' + cand_lst[0]

    webl['cand'] = webl['cand_name_lst'].map(refactor_str)
    cn['cand'] = cn['cand_name_lst'].map(refactor_str)

    df = pd.merge(left=webl, right=cn, on='CAND_ID', how='inner')
    df = df[[col for col in df.columns.values if ('_y' not in col)]]

    df.columns = df.columns.str.strip('_x')

    df['year'] = np.full(shape=(df.shape[0],), fill_value=year)

    return df

In [59]:
fec18 = wrangle_fec(webl18, cn18, 2018)
fec20 = wrangle_fec(webl20, cn20, 2020)
fec22 = wrangle_fec(webl22, cn22, 2022)
fec24 = wrangle_fec(webl24, cn24, 2024)
fec24.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,...,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP,year
0,H2AK01158,"PELTOLA, MARY",I,1,DEM,13443537.46,951851.88,14050828.27,0.00,691260.30,...,2024,H,C,C00812388,810 N STREET,SUITE 301,ANCHORAGE,AK,99501,2024
1,H2AK01083,"BEGICH, NICHOLAS III",C,2,REP,2810467.65,176570.41,2747371.58,17659.66,41233.99,...,2024,H,C,C00792341,PO BOX 671710,NaN,CHUGIAK,AK,99567,2024
2,H4AK00156,"DAHLSTROM, NANCY",C,2,REP,996163.60,435712.04,790351.61,0.00,0.00,...,2024,H,C,C00856716,PO BOX 242442,NaN,ANCHORAGE,AK,99524,2024
3,H4AL01255,"HOLMES, THOMAS BETHUNE MR.",C,1,DEM,17698.86,0.00,16817.50,0.00,0.00,...,2024,H,C,C00866939,"2117 CHARINGWOOD DRIVE WEST, MOBIL",NaN,MOBILE,AL,366952916,2024
4,H0AL01055,"CARL, JERRY LEE, JR",I,2,REP,2246839.19,547807.76,2631446.59,27316.59,453897.82,...,2024,H,C,C00697789,PO BOX 852138,NaN,MOBILE,AL,36685,2024


In [60]:
fec = pd.concat([fec18, fec20, fec22, fec24], axis=0)
fec.head()

,CAND_ID,CAND_NAME,CAND_ICI,PTY_CD,CAND_PTY_AFFILIATION,TTL_RECEIPTS,TRANS_FROM_AUTH,TTL_DISB,TRANS_TO_AUTH,COH_BOP,...,CAND_ELECTION_YR,CAND_OFFICE,CAND_STATUS,CAND_PCC,CAND_ST1,CAND_ST2,CAND_CITY,CAND_ST,CAND_ZIP,year
0,H8AK00132,"SHEIN, DIMITRI",C,1,DEM,209916.04,0.0,209574.16,0.0,0.00,...,2018,H,C,C00646521,PO BOX 90787,NaN,ANCHORAGE,AK,99509.0,2018
1,H6AK00045,"YOUNG, DONALD E",I,2,REP,1234680.31,0.0,1387687.05,0.0,269726.86,...,2018,H,C,C00012229,2504 FAIR BANKS ST,NaN,ANCHORAGE,AK,99503.0,2018
2,H8AK01031,"NELSON, THOMAS JOHN",C,2,REP,9288.48,0.0,8821.97,0.0,0.00,...,2018,H,C,C00681155,PO BOX 670123,NaN,CHUGIAK,AK,99567.0,2018
3,H8AK00140,"GALVIN, ALYSE",C,3,IND,1949643.68,154.7,1943398.59,0.0,0.00,...,2018,H,C,C00665711,P.O. BOX 90020,NaN,ANCHORAGE,AK,99509.0,2018
4,H8AL01066,"KENNEDY, ROBERT JR.",C,1,DEM,166845.21,0.0,166845.21,0.0,0.00,...,2018,H,C,C00667949,312-T SCHILLINGER RD #116,NaN,MOBILE,AL,36608.0,2018


In [61]:
fec24.shape, fec.shape

((2373, 42), (10519, 42))

In [55]:
fec.columns.values

array(['CAND_ID', 'CAND_NAME', 'CAND_ICI', 'PTY_CD',
       'CAND_PTY_AFFILIATION', 'TTL_RECEIPTS', 'TRANS_FROM_AUTH',
       'TTL_DISB', 'TRANS_TO_AUTH', 'COH_BOP', 'COH_COP', 'CAND_CONTRIB',
       'CAND_LOANS', 'OTHER_LOANS', 'CAND_LOAN_REPAY', 'OTHER_LOAN_REPAY',
       'DEBTS_OWED_BY', 'TTL_INDIV_CONTRIB', 'CAND_OFFICE_ST',
       'CAND_OFFICE_DISTRICT', 'SPEC_ELECTION', 'PRIM_ELECTION',
       'RUN_ELECTION', 'GEN_ELECTION', 'GEN_ELECTION_PRECENT',
       'OTHER_POL_CMTE_CONTRIB', 'POL_PTY_CONTRIB', 'CVG_END_DT',
       'INDIV_REFUNDS', 'CMTE_REFUNDS', 'cand_name_lst', 'cand',
       'CAND_ELECTION_YR', 'CAND_OFFICE', 'CAND_STATUS', 'CAND_PCC',
       'CAND_ST1', 'CAND_ST2', 'CAND_CITY', 'CAND_ST', 'CAND_ZIP'],
      dtype=object)